### CARGA DE LIBRERÍAS NECESARIAS PARA LA LIMPIEZA

In [1]:
import pandas as pd
import numpy as np
import os

### CARGA DEL ARCHIVO A LIMPIAR

In [2]:
Ruta_Archivo_Entrada = "../Original_Data/2025_3T_Flujosporentidadfederativa_actu__5__.xlsx"
Ruta_Archivo_Salida = '../Prepared_Data/State'
Archivo_Excel = pd.read_excel(Ruta_Archivo_Entrada, sheet_name=None, header=None)

In [3]:
'''Ubicación del entorno de trabajo y hojas de excel contenidas dentro del archivo'''
#print(os.getcwd())
Archivo_Excel.keys()

dict_keys(['Tipo de inversión', 'Origen', 'Actividad económica'])

### LIMPIEZA DE LA PRIMERA HOJA DEL ARCHIVO: 'Tipo de inversión'

##### Selección del archivo a limpiar 

In [4]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Investment_Types_by_State = Archivo_Excel['Tipo de inversión']
#display(Investment_Types)
Investment_Types_by_State_Description = Investment_Types_by_State.iloc[0]
#print(Investment_Description)
Investment_Types_by_State_Database = Investment_Types_by_State.iloc[-1,0]
#print(Investment_DataBase)

##### Limpieza general de dataframe

In [5]:
''' Se crea una copia del archivo con el cual trabajaremos '''
Investment_Types_by_State_1 = Investment_Types_by_State.copy()
''' Se elimninan las dos primeras filas y las tres últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Investment_Types_by_State_1 = Investment_Types_by_State_1.iloc[2:,:].reset_index(drop=True)
Investment_Types_by_State_1 = Investment_Types_by_State_1.iloc[:-3,:].reset_index(drop=True)
#display(Investment_Types_by_State_1)
'''
Dado el formato de origen en excel y el hecho de que algunas celdas estan combinadas, al cargar el archivo los valores contenidos en varias celdas no se han distribuido de
manera uniforme en las celdas de la misma fila en las que se contenia, por lo que al contenerse dentro de la celda más a la izquierda, se rellena hacía la derecha, esto 
nos devuelve las columnas con los años, donde antes se contenian y por último se combinan la celdas de los años y trimestres para obtener fechas completas.
'''
''' Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Investment_Types_by_State_1.iloc[0,1:] = Investment_Types_by_State_1.iloc[0,1:].ffill()
''' Se extraen los años contenidos en la fila 0 '''
Investment_Types_by_State_1_Años = Investment_Types_by_State_1.iloc[0,1:].astype(int).tolist()
#print(Investment_Types_by_State_1_Años)
''' Se extraen los trimestres contenidos en la fila 1 '''
Investment_Types_by_State_1_Trimestres = Investment_Types_by_State_1.iloc[1,1:].astype(int).tolist()
#print(Investment_Types_by_State_1_Trimestres)
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo en formato adecuado '''
Investment_Types_by_State_1_Fecha = [f"{a}Q{t}" for a, t in zip(Investment_Types_by_State_1_Años,Investment_Types_by_State_1_Trimestres)]
#print(Investment_Types_by_State_1_Fecha)
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Investment_Types_by_State_1.columns = [Investment_Types_by_State_1.iloc[0,0]] + Investment_Types_by_State_1_Fecha
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados'''
Investment_Types_by_State_1 = Investment_Types_by_State_1.iloc[2:,:].reset_index(drop=True)
#display(Investment_Types_by_State_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_38400\954700880.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Investment_Types_by_State_1.iloc[0,1:] = Investment_Types_by_State_1.iloc[0,1:].ffill()


##### Ingeniería de datos, métodos de categorización

In [6]:
'''
En esta sección aplicara ingeniería de datos, se categorizará por etiquetas basandonos en el nombre de los estados.
Se categorizará por etiqueta de estados, por lo tanto, se creará el diccionario con el nombre de dichos estados, esto será útil para representar información a través
de un formato largo y series temporales.
'''

Estados = ["Aguascalientes", "Baja California", "Baja California Sur", "Campeche", "Chiapas", "Chihuahua", "Ciudad de México", "Coahuila de Zaragoza",
           "Colima", "Durango", "Estado de México", "Guanajuato", "Guerrero","Hidalgo", "Jalisco", "Michoacán de Ocampo", "Morelos", "Nayarit",
           "Nuevo León", "Oaxaca", "Puebla", "Querétaro", "Quintana Roo", "San Luis Potosí", "Sinaloa", "Sonora", "Tabasco", "Tamaulipas",
           "Tlaxcala", "Veracruz de Ignacio de la Llave", "Yucatán", "Zacatecas"]

'''Se genera la copia del archivo con el cual trabajaremos'''
Investment_Types_by_State_2 = Investment_Types_by_State_1.copy()
'''A continuación se creará una nueva columna, esta contendra a los indicadores por estado'''
Investment_Types_by_State_2['Estado'] = ''
#Investment_Types_by_State_2
'''A continuación se sustituirán los valores en las columna de "Estado" '''
''' Rellenamos el valor de la columna Estado para la fila de Total general con ese mismo valor '''
Investment_Types_by_State_2.loc[0,'Estado'] = 'Total general' 
''' Copiamos el valor de la columna original "Entidad Federativa / Tipo de Inversión " si está en la lista de Estados creada anteriormente '''
Investment_Types_by_State_2.loc[Investment_Types_by_State_2['Entidad Federativa / Tipo de Inversión '].isin(Estados),'Estado'] = Investment_Types_by_State_2['Entidad Federativa / Tipo de Inversión ']
''' Reemplazamos los valores vacíos "" por valores nan en la columna "Estado"'''
Investment_Types_by_State_2['Estado'] = Investment_Types_by_State_2['Estado'].replace(["",False],np.nan)
''' Ahora rellenamos hacía abajo '''
Investment_Types_by_State_2['Estado'] = Investment_Types_by_State_2['Estado'].ffill(axis=0)
''' A continuación en la columna "Entidad Federativa / Tipo de Inversión" se sustituyen las celdas con el nombre de algún estado puesto que,
ya se ha agregado el identificador necesario'''
Investment_Types_by_State_2.loc[Investment_Types_by_State_2['Entidad Federativa / Tipo de Inversión '].isin(Estados),'Entidad Federativa / Tipo de Inversión '] = 'Total general por estado'
#display(Investment_Types_by_State_2.head(10))
'''Por último, se renombra la columna de "Entidad Federativa / Tipo de Inversión " por "Tipo de Inversión" para mayor legibilidad '''
Investment_Types_by_State_2 = Investment_Types_by_State_2.rename(columns={'Entidad Federativa / Tipo de Inversión ':'Tipo de Inversión'})
#display(Investment_Types_by_State_2.head(10))

##### Transformación de dataframe

In [7]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal además de un análisis categórico relativamente sencillo'''
Investment_Types_by_State_3 = pd.melt(
    Investment_Types_by_State_2,
    id_vars = ['Tipo de Inversión','Estado'],
    var_name = 'Año_Trimestre',
    value_name = 'IED'
)
'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Investment_Types_by_State_3['Fecha'] = pd.PeriodIndex(Investment_Types_by_State_3['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Investment_Types_by_State_3 = Investment_Types_by_State_3.drop(['Año_Trimestre'],axis=1)

#display(Investment_Types_by_State_3)

##### Formato general del archivo

In [8]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Investment_Types_by_State_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10062 entries, 0 to 10061
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Tipo de Inversión  10062 non-null  object        
 1   Estado             10062 non-null  object        
 2   IED                10062 non-null  object        
 3   Fecha              10062 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 314.6+ KB


##### Carga de datos al entorno local

In [9]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_1 = 'Investment_Type_by_State.csv'
Ruta_Salida_1 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_1)
Investment_Types_by_State_3.to_csv(Ruta_Salida_1, index=False, encoding='utf-8-sig')

### LIMPIEZA DE LA SEGUNDA HOJA DEL ARCHIVO: 'Origen'

##### Selección del archivo a limpiar 

In [10]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Country_Origin_Investment_by_State = Archivo_Excel['Origen']
#display(Country_Origin_Investment_by_State)
Origin_Investment_by_State_Description = Country_Origin_Investment_by_State.iloc[0,0]
#display(Origin_Investment_Description)
Origin_Investment_by_State_Database = Country_Origin_Investment_by_State.iloc[-1,0]
#display(Origin_Investment_Description)

##### Limpieza general de dataframe

In [11]:
''' Se crea una copia del archivo con el cual trabajaremos '''
Country_Origin_Investment_by_State_1 = Country_Origin_Investment_by_State.copy()
''' Se elimninan las dos primeras filas y las tres últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Country_Origin_Investment_by_State_1 = Country_Origin_Investment_by_State_1.iloc[2:,:].reset_index(drop=True)
Country_Origin_Investment_by_State_1 = Country_Origin_Investment_by_State_1.iloc[:-3,:].reset_index(drop=True)
#display(Country_Origin_Investment_by_State_1)
'''
Dado el formato de origen en excel y el hecho de que algunas celdas estan combinadas, al cargar el archivo los valores contenidos en varias celdas no se han distribuido de
manera uniforme en las celdas de la misma fila en las que se contenia, por lo que al contenerse dentro de la celda más a la izquierda, se rellena hacía la derecha, esto 
nos devuelve las columnas con los años, donde antes se contenian y por último se combinan la celdas de los años y trimestres para obtener fechas completas.
'''
''' Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Country_Origin_Investment_by_State_1.iloc[0,1:] = Country_Origin_Investment_by_State_1.iloc[0,1:].ffill()
''' Se extraen los años contenidos en la fila 0 '''
Country_Origin_Investment_by_State_1_Años = Country_Origin_Investment_by_State_1.iloc[0,1:].astype(int).tolist()
''' Se extraen los trimestres contenidos en la fila 1 '''
Country_Origin_Investment_by_State_1_Trimestres = Country_Origin_Investment_by_State_1.iloc[1,1:].astype(int).tolist()
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo con fechas en el formato adecuado '''
Country_Origin_Investment_by_State_1_Fecha = [f"{a}Q{t}" for a, t in zip(Country_Origin_Investment_by_State_1_Años, Country_Origin_Investment_by_State_1_Trimestres)]
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Country_Origin_Investment_by_State_1.columns = [Country_Origin_Investment_by_State_1.iloc[0,0]] + Country_Origin_Investment_by_State_1_Fecha
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados'''
Country_Origin_Investment_by_State_1 = Country_Origin_Investment_by_State_1.iloc[2:,:].reset_index(drop=True)
#display(Country_Origin_Investment_by_State_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_38400\1174902412.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Country_Origin_Investment_by_State_1.iloc[0,1:] = Country_Origin_Investment_by_State_1.iloc[0,1:].ffill()


##### Ingeniería de datos, métodos de categorización

In [12]:
'''
En esta sección aplicara ingeniería de datos, se categorizará por etiquetas basandonos en el nombre de los estados.
Se categorizará por etiqueta de estados, por lo tanto, se creará el diccionario con el nombre de dichos estados, esto será útil para representar información a través
de un formato largo y series temporales.
'''

Estados = ["Aguascalientes", "Baja California", "Baja California Sur", "Campeche", "Chiapas", "Chihuahua", "Ciudad de México", "Coahuila de Zaragoza",
           "Colima", "Durango", "Estado de México", "Guanajuato", "Guerrero","Hidalgo", "Jalisco", "Michoacán de Ocampo", "Morelos", "Nayarit",
           "Nuevo León", "Oaxaca", "Puebla", "Querétaro", "Quintana Roo", "San Luis Potosí", "Sinaloa", "Sonora", "Tabasco", "Tamaulipas",
           "Tlaxcala", "Veracruz de Ignacio de la Llave", "Yucatán", "Zacatecas"]

'''Se genera la copia del archivo con el cual trabajaremos'''
Country_Origin_Investment_by_State_2 = Country_Origin_Investment_by_State_1.copy()
'''A continuación se creará una nueva columna, esta contendra a los indicadores por estado'''
Country_Origin_Investment_by_State_2['Estado'] = ''
''' Rellenamos el valor de la columna Estado para la fila de Total general con ese mismo valor '''
Country_Origin_Investment_by_State_2.loc[0,'Estado'] = 'Total general'
''' Copiamos el valor de la columna original "Entidad Federativa / Origen" si está en la lista de Estados creada anteriormente '''
Country_Origin_Investment_by_State_2.loc[Country_Origin_Investment_by_State_2['Entidad Federativa / Origen'].isin(Estados), 'Estado'] = Country_Origin_Investment_by_State_2['Entidad Federativa / Origen']
''' Reemplazamos los valores vacíos "" por valores nan '''
Country_Origin_Investment_by_State_2['Estado'] = Country_Origin_Investment_by_State_2['Estado'].replace(['',False],np.nan)
''' Ahora rellenamos hacía abajo '''
Country_Origin_Investment_by_State_2['Estado'] = Country_Origin_Investment_by_State_2['Estado'].ffill(axis=0)
''' A continuación en la columna Entidad Federativa / Origen se sustituyen las celdas con el nombre de algún estado puesto que,
ya se ha agregado el identificador necesario'''
Country_Origin_Investment_by_State_2.loc[Country_Origin_Investment_by_State_2['Entidad Federativa / Origen'].isin(Estados),'Entidad Federativa / Origen'] = 'Total general por estado'
'''Por último, se renombra la columna de conceptos para mejor legibilidad'''
Country_Origin_Investment_by_State_2 = Country_Origin_Investment_by_State_2.rename(columns={'Entidad Federativa / Origen':'Inversión por país en cada estado'})
#display(Country_Origin_Investment_by_State_2)

##### Transformación de dataframe

In [13]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal'''
Country_Origin_Investment_by_State_3 = pd.melt(
    Country_Origin_Investment_by_State_2,
    id_vars = ['Inversión por país en cada estado','Estado'],
    var_name = 'Año_Trimestre',
    value_name = 'IED'
)
'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Country_Origin_Investment_by_State_3['Fecha'] = pd.PeriodIndex(Country_Origin_Investment_by_State_3['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Country_Origin_Investment_by_State_3 = Country_Origin_Investment_by_State_3.drop(['Año_Trimestre'],axis=1)
#display(Country_Origin_Investment_by_State_3)

##### Formato general del archivo

In [14]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Country_Origin_Investment_by_State_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155610 entries, 0 to 155609
Data columns (total 4 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   Inversión por país en cada estado  155610 non-null  object        
 1   Estado                             155610 non-null  object        
 2   IED                                155610 non-null  object        
 3   Fecha                              155610 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 4.7+ MB


##### Carga de datos al entorno local

In [15]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_2 = 'Investment_from_the_Countries_by_State.csv'
Ruta_Salida_2 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_2)
Country_Origin_Investment_by_State_3.to_csv(Ruta_Salida_2, index=False, encoding='utf-8-sig')

### LIMPIEZA DE LA TERCERA HOJA DEL ARCHIVO: 'Actividad económica'

##### Selección del archivo a limpiar 

In [16]:
'''Se selecciona la hoja de excel con la cual trabajaremos y obtenemos información general del archivo como su descripción y su fuente '''
Investment_by_Sector_in_the_States = Archivo_Excel['Actividad económica']
#display(Investment_by_Sector_in_the_States)
Investment_by_Sector_in_the_States_Description = Investment_by_Sector_in_the_States.iloc[0,0]
#print(Investment_by_Sector_in_the_States_Description)
Investment_by_Sector_in_the_States_Database = Investment_by_Sector_in_the_States.iloc[-1,0]
#print(Investment_by_Sector_in_the_States_Database)

##### Limpieza general de dataframe

In [17]:
''' Se crea una copia del archivo con el cual trabajaremos '''
Investment_by_Sector_in_the_States_1 = Investment_by_Sector_in_the_States.copy()
''' Se elimninan las dos primeras filas y las cuatro últimas filas del dataframe, pues contienen información no relevante para el proyecto y alteran el formato a analizar '''
Investment_by_Sector_in_the_States_1 = Investment_by_Sector_in_the_States_1.iloc[2:,:].reset_index(drop=True)
Investment_by_Sector_in_the_States_1 = Investment_by_Sector_in_the_States_1.iloc[:-4,:].reset_index(drop=True)
'''
Dado el formato de origen en excel y el hecho de que algunas celdas estan combinadas, al cargar el archivo los valores contenidos en varias celdas no se han distribuido de
manera uniforme en las celdas de la misma fila en las que se contenia, por lo que al contenerse dentro de la celda más a la izquierda, se rellena hacía la derecha, esto 
nos devuelve las columnas con los años, donde antes se contenian y por último se combinan la celdas de los años y trimestres para obtener fechas completas.
'''
''' Se rellena hacía la derecha asegurando rellenar con los años adecuados las celdas que correspondan al mismo año de su formato original, esto en la fila 0 '''
Investment_by_Sector_in_the_States_1.iloc[0,1:] = Investment_by_Sector_in_the_States_1.iloc[0,1:].ffill()
''' Se extraen los años contenidos en la fila 0 '''
Investment_by_Sector_in_the_States_1_Años = Investment_by_Sector_in_the_States_1.iloc[0,1:].astype(int).tolist()
#print(Investment_by_Sector_in_the_States_1_Años)
''' Se extraen los trimestres contenidos en la fila 1 '''
Investment_by_Sector_in_the_States_1_Trimestres = Investment_by_Sector_in_the_States_1.iloc[1,1:].astype(int).tolist()
#print(Investment_by_Sector_in_the_States_1_Trimestres)
''' Se crea un arreglo uniendo años y trimestres para así tener un arreglo con fechas en el formato adecuado '''
Investment_by_Sector_in_the_States_1_Fecha = [f"{a}Q{t}" for a, t  in zip(Investment_by_Sector_in_the_States_1_Años,Investment_by_Sector_in_the_States_1_Trimestres)]
#print(Investment_by_Sector_in_the_States_1_Fecha)
''' Se sustituyen los valores compuestos como nombres de columnas junto con el valor que tenía por defecto la fila [0,0] '''
Investment_by_Sector_in_the_States_1.columns = [Investment_by_Sector_in_the_States_1.iloc[0,0]] + Investment_by_Sector_in_the_States_1_Fecha
''' Se ajusta nuevamente el dataframe, esto con la finalidad de eliminar las filas con la información de años y trimestres pues ya se encuentran como encabezados '''
Investment_by_Sector_in_the_States_1 = Investment_by_Sector_in_the_States_1.iloc[2:,:].reset_index(drop=True)
#display(Investment_by_Sector_in_the_States_1)

C:\Users\adnac\AppData\Local\Temp\ipykernel_38400\1741709152.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Investment_by_Sector_in_the_States_1.iloc[0,1:] = Investment_by_Sector_in_the_States_1.iloc[0,1:].ffill()


##### Ingeniería de datos, métodos de categorización y jerarquización 

In [18]:
'''Se genera la copia del archivo con el cual trabajaremos'''
Investment_by_Sector_in_the_States_2 = Investment_by_Sector_in_the_States_1.copy()
'''Dado el formato de los datos, se jerarquizan los conceptos contenidos dentro de la columna "Entidad Federativa / Actividad Económica" con el fin de separar tanto de manera jerarquica como categórica los datos'''
Investment_by_Sector_in_the_States_2[['Código','Concepto']] = Investment_by_Sector_in_the_States_2['Entidad Federativa / Actividad Económica'].str.split(r"\s+",n=1,expand=True)
Investment_by_Sector_in_the_States_2['Nivel'] = Investment_by_Sector_in_the_States_2['Código'].apply(lambda x: 2 if '-' in x else len(x))
Investment_by_Sector_in_the_States_2['Sector'] = Investment_by_Sector_in_the_States_2['Código'].apply(lambda x: x if '-' in x else x[:2])
Investment_by_Sector_in_the_States_2['Subsector'] = Investment_by_Sector_in_the_States_2['Código'].apply(lambda x: None if '-' in x or len(x)<3 else x[:3])
Investment_by_Sector_in_the_States_2['Rama'] = Investment_by_Sector_in_the_States_2['Código'].apply(lambda x: None if '-' in x or len(x)<4 else x[:4])
Investment_by_Sector_in_the_States_2.loc[0,'Concepto'] = 'Total general' 
Investment_by_Sector_in_the_States_2.loc[0,'Nivel':] = 1

'''Se categorizará los datos por estado para un análisis más detallado'''
Estados = ["Aguascalientes", "Baja California", "Baja California Sur", "Campeche", "Chiapas", "Chihuahua", "Ciudad de México", "Coahuila de Zaragoza",
           "Colima", "Durango", "Estado de México", "Guanajuato", "Guerrero","Hidalgo", "Jalisco", "Michoacán de Ocampo", "Morelos", "Nayarit",
           "Nuevo León", "Oaxaca", "Puebla", "Querétaro", "Quintana Roo", "San Luis Potosí", "Sinaloa", "Sonora", "Tabasco", "Tamaulipas",
           "Tlaxcala", "Veracruz de Ignacio de la Llave", "Yucatán", "Zacatecas"]
'''A continuación se creará una nueva columna, esta contendra a los indicadores por estado'''
Investment_by_Sector_in_the_States_2['Estado'] = ''
''' Rellenamos el valor de la columna Estado para la fila de Total general con ese mismo valor '''
Investment_by_Sector_in_the_States_2.loc[0,'Estado'] = 'Total general'
''' Copiamos el valor de la columna original "Entidad Federativa / Origen" si está en la lista de Estados creada anteriormente '''
Investment_by_Sector_in_the_States_2.loc[Investment_by_Sector_in_the_States_2['Entidad Federativa / Actividad Económica'].isin(Estados),'Estado'] = Investment_by_Sector_in_the_States_2['Entidad Federativa / Actividad Económica']
''' Reemplazamos los valores vacíos "" por valores nan '''
Investment_by_Sector_in_the_States_2['Estado'] = Investment_by_Sector_in_the_States_2['Estado'].replace(['',False],np.nan)
''' Ahora rellenamos hacía abajo '''
Investment_by_Sector_in_the_States_2['Estado'] = Investment_by_Sector_in_the_States_2['Estado'].ffill(axis=0)
'''Reemplazamos los valores contenidos dentro de la columna "Entidad Federativa / Actividad Económica" que son nombres de estados con un indicador general, esto ayudará a crear filtrar datos más adelante '''
Investment_by_Sector_in_the_States_2.loc[Investment_by_Sector_in_the_States_2['Entidad Federativa / Actividad Económica'].isin(Estados),['Entidad Federativa / Actividad Económica','Concepto']] = 'Total general por estado'
'''Agregamos un identificador general para los totales generales por estado, esto ayudará a identificar y filtrar datos'''
Investment_by_Sector_in_the_States_2.loc[Investment_by_Sector_in_the_States_2['Concepto'] == 'Total general por estado',['Nivel','Sector','Subsector','Rama']] = 0
'''Eliminamos las columnas que ya no son necesarias '''
Investment_by_Sector_in_the_States_2 = Investment_by_Sector_in_the_States_2.drop(['Código','Entidad Federativa / Actividad Económica'],axis=1)
#display(Investment_by_Sector_in_the_States_2)

##### Transformación del dataframe

In [19]:
'''Para analizar la información en este proyecto se hará uso de un formato largo, pues permite un análisis temporal además de un análisis categórico relativamente sencillo'''
Investment_by_Sector_in_the_States_3 = pd.melt(
    Investment_by_Sector_in_the_States_2,
    id_vars = ['Concepto','Estado','Nivel','Sector','Subsector','Rama'],
    var_name = 'Año_Trimestre',
    value_name = 'IED'
)
'''A continuación se convertirá la columna Año_Trimestre a un formato adecuado de fecha, se pasará de un formato YYYYQn -> YYYYMMD para, posteriormente eliminar la columna Año_Trimestre '''
Investment_by_Sector_in_the_States_3['Fecha'] = pd.PeriodIndex(Investment_by_Sector_in_the_States_3['Año_Trimestre'],freq='Q').to_timestamp(how='end').normalize()
Investment_by_Sector_in_the_States_3 = Investment_by_Sector_in_the_States_3.drop(['Año_Trimestre'],axis=1)
#display(Investment_by_Sector_in_the_States_3)

##### Formato general del archivo

In [20]:
'''A continuación se presenta un análisis general básico que permita identificar posibles inconsistencias en los datos antes de proceder al guardado de la información y con análisis los análisis de cualquier tipo'''
Investment_by_Sector_in_the_States_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 659178 entries, 0 to 659177
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Concepto   659178 non-null  object        
 1   Estado     659178 non-null  object        
 2   Nivel      659178 non-null  int64         
 3   Sector     659178 non-null  object        
 4   Subsector  614562 non-null  object        
 5   Rama       431496 non-null  object        
 6   IED        659178 non-null  object        
 7   Fecha      659178 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 40.2+ MB


##### Carga de datos al entorno local

In [21]:
'''El siguiente código muestra únicamente el proceso de guardado de nuestro archivo procesado, asegurando un nombre facilmente identificable,
junto con un formato adecuado al idioma español latino '''
Nombre_Archivo_Hoja_3 = 'Investment_by_Sector_from_the_States.csv'
Ruta_Salida_3 = os.path.join(Ruta_Archivo_Salida,Nombre_Archivo_Hoja_3)
Investment_by_Sector_in_the_States_3.to_csv(Ruta_Salida_3, index=False, encoding='utf-8-sig')